## Cell 0: Fix CUDA + verify inputs

In [1]:

import subprocess

# Check what CUDA version the GPU actually needs
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:400])

# Reinstall PyTorch with correct CUDA version
!pip uninstall torch torchvision torchaudio -y -q
!pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu126 -q

# DON'T reload torch - just restart the kernel or use a clean import
# Instead, let's verify after a kernel restart or use a different approach
print("\n⚠️ PyTorch reinstalled. RESTART THE KERNEL before proceeding!")
print("   Go to: Kernel → Restart Kernel")

Sat Apr 18 22:07:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 83.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 86.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 228.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 116.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 241.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 227.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 143.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 1 : Install & Imports

In [3]:
!pip install ultralytics pandas matplotlib opencv-python-headless seaborn -q

import os, gc, glob, random, warnings
import cv2, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
from ultralytics import YOLO
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT    = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

print("✅ Libraries ready")
print(f"   Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Verify Notebook 1 outputs exist
required = ['flame2_labels.csv', 'frame_split.npz', 'data.yaml']
for r in required:
    path = f'{OUT}/{r}'
    if os.path.exists(path):
        print(f"✅ Found: {r}")
    else:
        print(f"❌ Missing: {r}  → Run Notebook 1 first!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Libraries ready
   Device : cuda
   GPU    : Tesla P100-PCIE-16GB
   VRAM   : 17.1 GB
✅ Found: flame2_labels.csv
✅ Found: frame_split.npz
✅ Found: data.yaml


## Cell 2: Train YOLOv11s

In [4]:
torch.cuda.empty_cache(); gc.collect()

yaml_path  = f'{OUT}/data.yaml'
TRAIN_DIR  = '/kaggle/working/yolo11_fire_smoke'

print("🚀 Starting YOLOv11s training...")
print("   Fixed hyperparameters vs original broken code:")
print("   lr0=0.001 (was 0.01) | box=5.0 (was 7.5) | warmup=5 (was 3)")
print("="*55)

model = YOLO('yolo11s.pt')

results = model.train(
    data            = yaml_path,
    epochs          = 100,
    imgsz           = 640,
    batch           = 16,
    workers         = 4,
    device          = 0,

    # ── Optimisation (stable — no EMA NaN) ──────────────────
    optimizer       = 'AdamW',
    lr0             = 0.001,       
    lrf             = 0.01,
    momentum        = 0.937,
    weight_decay    = 0.0005,
    warmup_epochs   = 5,           
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,

    # ── Loss weights (prevent DFL blow-up) ──────────────────
    box             = 5.0,         
    cls             = 0.5,
    dfl             = 1.5,
    dropout         = 0.1,

    # ── Augmentation ────────────────────────────────────────
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    shear=2.0, flipud=0.0, fliplr=0.5,
    mosaic     = 0.5,         
    mixup      = 0.0,         
    copy_paste = 0.0,
    
    # ── Stopping & saving ───────────────────────────────────
    patience        = 15,
    save            = True,
    save_period     = 5,
    val             = True,
    plots           = True,
    verbose         = True,

    project         = TRAIN_DIR,
    name            = 'train_v2',
    exist_ok        = True,
)

print("\n✅ Training complete!")

# Copy best model to outputs
import shutil
best_src = f'{TRAIN_DIR}/train_v2/weights/best.pt'
best_dst = f'{OUT}/best_yolo.pt'
shutil.copy(best_src, best_dst)
print(f"💾 Best model saved → {best_dst}")


🚀 Starting YOLOv11s training...
   Fixed hyperparameters vs original broken code:
   lr0=0.001 (was 0.01) | box=5.0 (was 7.5) | warmup=5 (was 3)
Ultralytics 8.4.39 🚀 Python-3.12.12 torch-2.11.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/outputs/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo